# 03 — Data Cleaning
## Greater London House Price Analysis

Este notebook documenta el proceso de limpieza del dataset **UK Price Paid** 
filtrado para Greater London.

### Objetivo
Analizar la calidad del dato en la tabla Silver, justificar las decisiones 
de limpieza y tratar los valores nulos de forma apropiada.

### Tabla de entrada
`workspace.default.london_silver` — 2,384,979 filas · 21 columnas

### Pipeline
```
Bronze (raw) → Silver (limpia) → Silver enriquecida (nulos tratados)
```


In [0]:
from pyspark.sql.functions import col, count, when, round as spark_round

df_silver = spark.table("workspace.default.london_silver")

total = df_silver.count()
print(f"Total filas cargadas: {total:,}")
print(f"Total columnas: {len(df_silver.columns)}")

## 1. Análisis de Nulos

Inspeccionamos cuántos valores nulos tiene cada columna antes de tomar decisiones.

In [0]:
print(f"{'Columna':<25} {'Nulos':>10} {'% Nulos':>10}")
print("-" * 50)

for c in df_silver.columns:
    nulos = df_silver.filter(col(c).isNull()).count()
    pct = round(nulos / total * 100, 2)
    print(f"{c:<25} {nulos:>10,} {pct:>9.2f}%")

## 2. Decisiones de Limpieza

### Columnas con 0 nulos ✅
Todas las columnas tienen 0 nulos tras el proceso de limpieza aplicado
en los notebooks anteriores y en este notebook.

### Nulos identificados en Bronze y tratados en el pipeline

| Columna | Nulos en Bronze | % | Decisión aplicada | Justificación |
|---------|----------------|---|-------------------|---------------|
| `saon` | ~1.5M | 63% | Rellenado con `"N/A"` | Campo opcional — solo existe en edificios con múltiples unidades |
| `locality` | ~1.78M | 75% | Rellenado con `"UNKNOWN"` | Campo administrativo secundario, no requerido oficialmente |
| `postcode` | ~4,748 | 0.2% | Rellenado con `"UNKNOWN"` | No usada en el modelo pero conviene no dejar nulos |
| `street` | ~356 | 0.01% | Rellenado con `"UNKNOWN"` | Campo de dirección, no relevante para el análisis |
| `paon` | ~23 | 0.001% | Rellenado con `"N/A"` | Número de casa, campo de dirección opcional |

> Los valores de nulos en Bronze fueron identificados en el EDA local
> y confirmados en el notebook `02_data_exploration`.

In [0]:
from pyspark.sql.functions import coalesce, lit

df_clean = df_silver \
    .fillna("N/A",      subset=["saon", "paon"]) \
    .fillna("UNKNOWN",  subset=["locality", "postcode", "street"])

print("=== NULOS DESPUÉS DEL TRATAMIENTO ===")
print(f"\n{'Columna':<25} {'Nulos antes':>12} {'Nulos después':>14}")
print("-" * 55)

columnas_tratadas = ["saon", "paon", "locality", "postcode", "street"]
for c in columnas_tratadas:
    antes  = df_silver.filter(col(c).isNull()).count()
    despues = df_clean.filter(col(c).isNull()).count()
    print(f"{c:<25} {antes:>12,} {despues:>14,}")

## 3. Verificación de Columnas Críticas

Confirmamos que las columnas usadas en el análisis Gold y en el modelo 
predictivo no tienen ningún nulo.

In [0]:
columnas_criticas = [
    "price", "date_of_transfer", "district", "property_type",
    "year", "month", "quarter", "property_type_desc", "price_range"
]

print("Columnas críticas para el análisis:\n")
todos_ok = True
for c in columnas_criticas:
    nulos = df_clean.filter(col(c).isNull()).count()
    estado = "0 nulos" if nulos == 0 else f"{nulos:,} nulos"
    print(f"  {c:<25} {estado}")
    if nulos > 0:
        todos_ok = False

print(f"\n{'Dataset listo para Gold' if todos_ok else 'Revisar columnas con nulos'}")

In [0]:
from pyspark.sql.functions import quarter, when, year, month

# Reconstruir columnas calculadas que están vacías
df_clean = df_clean \
    .withColumn("quarter", quarter(col("date_of_transfer"))) \
    .withColumn("property_type_desc",
        when(col("property_type") == "D", "Detached")
        .when(col("property_type") == "S", "Semi-Detached")
        .when(col("property_type") == "T", "Terraced")
        .when(col("property_type") == "F", "Flat")
        .otherwise("Other")
    ) \
    .withColumn("price_range",
        when(col("price") < 250000, "Bajo (<250k)")
        .when(col("price") < 500000, "Medio (250k-500k)")
        .when(col("price") < 1000000, "Alto (500k-1M)")
        .otherwise("Premium (>1M)")
    )

print("Columnas reconstruidas correctamente")
df_clean.select("quarter", "property_type_desc", "price_range").show(5)

## 4. Guardar tabla Silver final

In [0]:
df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.london_silver")

print("Tabla london_silver actualizada con nulos tratados")
print(f"   Filas totales:  {df_clean.count():,}")
print(f"   Columnas:       {len(df_clean.columns)}")

## 5. Resumen del proceso de limpieza

| Etapa | Filas entrada | Filas salida | Filas eliminadas |
|-------|--------------|--------------|-----------------|
| Bronze → Silver (filtro precio) | 3,929,180 | 2,384,979 | 1,544,201 |
| Silver → Silver limpia (nulos) | 2,384,979 | 2,384,979 | 0 |

### Decisiones clave
- **Filtro de precio:** eliminados registros < £10,000 y > £10,000,000
- **Filtro de fechas:** rango 2005–2025 (21 años de datos)
- **Nulos:** tratados con valores por defecto, sin eliminar filas
- **Duplicados:** 0 encontrados — no fue necesaria ninguna acción

### Tabla final
`workspace.default.london_silver` — **2,384,979 filas · 21 columnas** ✅